# 04 — Tuning recuperable de tres familias
Mismos datos, features y tres folds temporales; sin hold-out. 30 configuraciones diferentes por familia, 90 en total. Grilla exhaustiva acotada: equivalente a una busqueda de hiperparametros completa en espacios pequenos, sin muestrear duplicados.

Logistica: 15 C x 2 class_weight. MLP: 5 alpha x 3 arquitecturas x 2 learning_rate_init. RF: 5 profundidades x 3 min_samples_leaf x 2 max_features. Estas dimensiones controlan regularizacion, capacidad y optimizacion; n_estimators/iteraciones se mantienen acotados. Cada configuracion es hija del padre de su modelo y tiene tres folds hijos. Dos hilos y RF n_jobs=2. Se recuperan configuraciones terminadas por identidad/hash y estado MLflow.

El intento TabPFN no produjo una media de tres folds y no entra en un ranking comparable. Se conservan las siete corridas iniciales y todos los intentos, sin contarlos como 90 configuraciones.


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/cierre.py").is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import display
import pandas as pd


In [2]:
from src.cierre import tune_all
tuning = tune_all()
display(tuning[['candidate','default_auc','default_sd','tuned_auc','tuned_sd','delta','run_id']])
plan = json.loads((ROOT / 'docs/cierre/tuning_plan.json').read_text())
print(json.dumps(plan, indent=2))

C:\Users\PC\Documents\Codex\2026-11-10\pset-2-verificar-conexi-n-con\pset_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/17 22:45:51 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\PC\Documents\Codex\2026-11-10\pset-2-verificar-conexi-n-con\pset_3\.venv\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


90 configuraciones terminadas verificadas: recuperadas sin reentrenar.


,candidate,default_auc,default_sd,tuned_auc,tuned_sd,delta,run_id
0,logistica,0.570464,0.002261,0.570509,0.002199,0.000045,8ab70273357640a4b2a8e5719a448704
1,mlp,0.568070,0.007671,0.569234,0.005586,0.001164,1c32b608c7024455bbe68074121cf915
2,random_forest,0.555783,0.007920,0.565762,0.005920,0.009979,673a4d46ad944117b869f078c396ebe5


{
  "identity": "f0bfa3d78673d26729138a0848fdfb3cb59f21999737077058db08610b287b52",
  "models": [
    "logistica",
    "mlp",
    "random_forest"
  ],
  "primary_metric": "roc_auc",
  "n_folds": 3,
  "method": "exhaustive bounded grid (30 distinct configurations/model)",
  "spaces": {
    "logistica": {
      "model__C": [
        0.0001,
        0.0003,
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3,
        0.5,
        1.0,
        2.0,
        3.0,
        10.0,
        30.0,
        100.0
      ],
      "model__class_weight": [
        null,
        "balanced"
      ]
    },
    "mlp": {
      "model__alpha": [
        0.0001,
        0.001,
        0.01,
        0.1,
        1.0
      ],
      "model__hidden_layer_sizes": [
        [
          16
        ],
        [
          32,
          16
        ],
        [
          64
        ]
      ],
      "model__learning_rate_init": [
        0.0005,
        0.001
      ]
    },
    "random_forest

In [3]:
configs = pd.read_csv(ROOT / 'docs/cierre/tuning_configuraciones.csv')
assert len(configs) >= 90
display(configs.groupby('candidate').size().rename('configuraciones'))
assert configs.config_key.nunique() == len(configs)
print('Hold-out utilizado:', False)

candidate
logistica        30
mlp              30
random_forest    30
Name: configuraciones, dtype: int64

Hold-out utilizado: False


Las diferencias de medias se reportan junto con la desviacion. No constituyen una prueba de mejora significativa. No elegir otra metrica despues de ver el ranking. Los mejores parametros y predicciones OOF quedan persistidos para congelar la decision sin volver a ajustar CV.